# Video & 3D generation — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normal_pdf(x,mu,sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2)/(sigma*np.sqrt(2*np.pi))
def softmax(a):
    a=np.asarray(a,dtype=float); e=np.exp(a-a.max()); return e/e.sum()
def mse(a,b): return float(np.mean((np.asarray(a)-np.asarray(b))**2))
print('setup ok')

## temporal and geometric consistency
Diffusion and flow matching provide generative engines, while geometry, projection, and sequence modeling add the constraints that still images do not need. ControlNet (10.15) and latent diffusion (10.14) become building blocks for camera, depth, and motion control.

In [ ]:
# 1) a 3D point follows a linear trajectory
p0=np.array([0.,0.,0.]); v=np.array([1.,.5,.2]); ts=np.arange(5); pts=p0+ts[:,None]*v
plt.figure(figsize=(5,3)); plt.plot(ts,pts); plt.title('3D coordinates over time')
assert np.allclose(pts[3],[3,1.5,.6])

In [ ]:
# 2) projecting 3D motion creates 2D frame positions
proj=pts[:,:2]/(1+0.1*pts[:,2:3])
plt.figure(figsize=(4,4)); plt.plot(proj[:,0],proj[:,1],'o-'); plt.title('projected camera track')
assert proj.shape==(5,2)

In [ ]:
# 3) independent frame noise causes flicker
rng=np.random.default_rng(0); noisy=proj+rng.normal(0,.08,proj.shape); jitter=np.linalg.norm(np.diff(noisy-proj,axis=0),axis=1)
plt.figure(figsize=(5,3)); plt.plot(jitter,'o-'); plt.title('frame-to-frame jitter')
assert jitter.mean()>0

In [ ]:
# 4) temporal smoothing reduces flicker
smooth=noisy.copy(); smooth[1:-1]=(noisy[:-2]+noisy[1:-1]+noisy[2:])/3; jitter2=np.linalg.norm(np.diff(smooth-proj,axis=0),axis=1)
plt.figure(figsize=(5,3)); plt.plot(jitter,label='raw'); plt.plot(jitter2,label='smooth'); plt.legend(); plt.title('temporal consistency reduces jitter')
assert jitter2.mean()<jitter.mean()

In [ ]:
# 5) two views share the same underlying 3D points
view1=pts[:,:2]; view2=pts[:,[2,1]]
plt.figure(figsize=(4,4)); plt.plot(view1[:,0],view1[:,1],'o-',label='view 1'); plt.plot(view2[:,0],view2[:,1],'s-',label='view 2'); plt.legend(); plt.title('different projections, shared 3D path')
assert np.allclose(view1[:,1],view2[:,1])